In [ ]:
!pip install datasets pandas scikit-learn

## Carregar os Datasets

In [11]:
from datasets import load_dataset

hc3 = load_dataset(
    'Hello-SimpleAI/HC3', 
    revision='refs/convert/parquet'
)

openturingbench = load_dataset(
    'MLNTeam-Unical/OpenTuringBench', 
    revision='refs/convert/parquet'
)

claude3_opus = load_dataset(
    'nothingiisreal/Claude-3-Opus-Instruct-15K', 
    'Instruct Data v2 - Merged'
)

authentitext = load_dataset(
    'csv',
    data_files='data/AuthentiText_X_2026_AI_vs_Human_Detection_1K.csv'
)

print(hc3['train'][0])
print(openturingbench['train'][0])
print(claude3_opus['train'][0])
print(authentitext['train'][0])

{'id': '0', 'question': 'Why is every book I hear about a " NY Times # 1 Best Seller " ? ELI5 : Why is every book I hear about a " NY Times # 1 Best Seller " ? Should n\'t there only be one " # 1 " best seller ? Please explain like I\'m five.', 'human_answers': ['Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you \'re still an " oscar - winning " film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n\'t stay there for long , you still achieved the status . Hence , # 1 best seller .', "If you 're hearing about it , it 's because it was a very good or very well - publicized book ( or both ) , and almost every good 

## Processar e uniformizar datasets

In [12]:
import pandas as pd

# Lista vazia que vai guardar todas as linhas uniformizadas
dados_finais = []

# ---------------------------------------------------------
# 1. Processar HC3
# ---------------------------------------------------------
for linha in hc3['train']:
    linha = dict(linha) 
    # Textos humanos
    for resposta_humana in linha['human_answers']:
        dados_finais.append({
            'Text': resposta_humana, 
            'Label': 'Human'
        })
    
    # Textos da IA (ChatGPT)
    for resposta_ai in linha['chatgpt_answers']:
        dados_finais.append({
            'Text': resposta_ai, 
            'Label': 'ChatGPT'
        })

# ---------------------------------------------------------
# 2. Processar OpenTuringBench
# ---------------------------------------------------------
for linha in openturingbench['train']:
    linha = dict(linha) 

    modelo = linha['model']
    if 'llama' in modelo.lower():
        modelo = 'Meta'
        
    dados_finais.append({
        'Text': linha['content'], 
        'Label': modelo
    })

# ---------------------------------------------------------
# 3. Processar Claude-3-Opus
# ---------------------------------------------------------
for linha in claude3_opus['train']:
    linha = dict(linha) 

    dados_finais.append({
        'Text': linha['response'],
        'Label': 'Anthropic'
    })

# ---------------------------------------------------------
# 4. Processar AuthentiText
# ---------------------------------------------------------
for linha in authentitext['train']:
    linha = dict(linha) 
    # Se author_type for Humano, o Label é 'Human', senão usamos o model_source
    if linha['author_type'] == 'AI':
        label = linha['model_source']
    else:
        label = 'Human'
        
    dados_finais.append({
        'Text': linha['content_text'], 
        'Label': label
    })

# ---------------------------------------------------------
# JUNTAR, FORMATAR E GUARDAR
# ---------------------------------------------------------
# Converter para DataFrame
df = pd.DataFrame(dados_finais)

# Shuffle dos dados para misturar as diferentes origens
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

labels_unicos = df['Label'].unique()
print("Labels :", labels_unicos)

Labels : <ArrowStringArray>
[          'Qwen/Qwen2.5-7B-Instruct',                               'Meta',
                              'Human',                            'ChatGPT',
          'Intel/neural-chat-7b-v3-3', 'mistralai/Mistral-7B-Instruct-v0.3',
  'upstage/SOLAR-10.7B-Instruct-v1.0',               'google/gemma-2-9b-it',
    'microsoft/Phi-3.5-mini-instruct',                          'Anthropic',
                             'Gemini',                              'GPT-4',
                             'Claude']
Length: 13, dtype: str


## Uniformizar Labels

In [13]:
# 1. Criar o dicionário de mapeamento (o que queremos manter e como se vai chamar)
mapeamento_modelos = {
    'Meta': 'Meta',
    'google/gemma-2-9b-it': 'Google',
    'Gemini': 'Google',
    'Anthropic': 'Anthropic',
    'Claude': 'Anthropic',
    'ChatGPT': 'OpenAI',
    'GPT-4': 'OpenAI',
    'Human': 'Human'
}

# 2. Aplicar o mapeamento à coluna 'Label'
# O que não estiver no dicionário (Qwen, Intel, Mistral, etc.) vai ficar 'NaN'
df['Label'] = df['Label'].map(mapeamento_modelos)

# 3. Remover todas as linhas que ficaram com 'NaN' (ou seja, apagar os modelos indesejados)
df = df.dropna(subset=['Label'])

# 4. Confirmar que a limpeza funcionou
print("Labels finais após a limpeza:")
print(df['Label'].unique())

print("\nQuantidade de textos por Label:")
print(df['Label'].value_counts())

Labels finais após a limpeza:
<ArrowStringArray>
['Meta', 'Human', 'OpenAI', 'Google', 'Anthropic']
Length: 5, dtype: str

Quantidade de textos por Label:
Label
Human        117739
OpenAI        53917
Google        33268
Meta          33140
Anthropic      9566
Name: count, dtype: int64


## Reduzir Dataframe

In [14]:
df= df.sample(frac=0.1, random_state=42)
print(df['Label'].value_counts())

Label
Human        11852
OpenAI        5296
Google        3345
Meta          3308
Anthropic      962
Name: count, dtype: int64


## Modelo Numpy das aulas

In [ ]:
import numpy as np
from collections import Counter

import sys
import os
sys.path.append(os.path.abspath("numpy_model"))
from numpy_model.neuralnet import NeuralNetwork
from numpy_model.layers import DenseLayer
from numpy_model.activation import ReLUActivation, SoftmaxActivation
from numpy_model.losses import CategoricalCrossEntropy
from numpy_model.metrics import accuracy
from numpy_model.data import Data


# -----------------------------------------------------------------
# 1. PREPARAÇÃO DOS LABELS (Y_treino)
# -----------------------------------------------------------------
# Mapeamento para garantir a ordem dos pesos: [Human, OpenAI, Google, Meta, Anthropic]
label_map = {'Human': 0, 'OpenAI': 1, 'Google': 2, 'Meta': 3, 'Anthropic': 4}
y_indices = df['Label'].map(label_map).values

# Criar One-Hot Encoding (exigido pela CategoricalCrossEntropy)
num_classes = 5
Y_treino = np.eye(num_classes)[y_indices]

# -----------------------------------------------------------------
# 2. VETORIZAÇÃO DO TEXTO (X_treino) - Bag of Words Manual
# -----------------------------------------------------------------
print("A criar vocabulário...")
todas_palavras = " ".join(df['Text']).lower().split()
# Selecionar as 1000 palavras mais frequentes como "features"
vocab = [palavra for palavra, count in Counter(todas_palavras).most_common(1000)]
word_to_idx = {palavra: i for i, palavra in enumerate(vocab)}

def vetorizar_texto(texto):
    vetor = np.zeros(len(vocab))
    for palavra in str(texto).lower().split():
        if palavra in word_to_idx:
            vetor[word_to_idx[palavra]] += 1
    return vetor

print("A converter textos em vetores (X_treino)...")
X_treino = np.array([vetorizar_texto(t) for t in df['Text']])

# -----------------------------------------------------------------
# 3. CONFIGURAR E TREINAR O MODELO
# -----------------------------------------------------------------
# Pesos calculados para lidar com o desbalanceamento
pesos = [0.42, 0.91, 1.48, 1.49, 5.17] 

modelo = NeuralNetwork(
    epochs=50, 
    batch_size=64, 
    learning_rate=0.01,
    loss=CategoricalCrossEntropy(class_weights=pesos), 
    metric=accuracy, 
    verbose=True
)

# Adicionar camadas
num_features = X_treino.shape[1]
modelo.add(DenseLayer(n_units=128, input_shape=(num_features,)))
modelo.add(ReLUActivation())

modelo.add(DenseLayer(n_units=64))
modelo.add(ReLUActivation())

# Camada final: 5 neurónios para as 5 classes
modelo.add(DenseLayer(n_units=num_classes))
modelo.add(SoftmaxActivation())

# Criar o objeto Data esperado pelo modelo
dataset_treino = Data(X=X_treino, y=Y_treino)

# Treinar o modelo (Método fit definido em neuralnet.py)
modelo.fit(dataset_treino)

# Fazer previsões e avaliar
previsoes = modelo.predict(dataset_treino)
print("Treino concluído!")

A criar vocabulário...
A converter textos em vetores (X_treino)...
Epoch 1/50 - loss: 6.7720 - accuracy: 0.6493
Epoch 2/50 - loss: 0.7714 - accuracy: 0.7513
Epoch 3/50 - loss: 0.6396 - accuracy: 0.7923
Epoch 4/50 - loss: 0.5560 - accuracy: 0.8159
Epoch 5/50 - loss: 0.5039 - accuracy: 0.8338
Epoch 6/50 - loss: 0.4627 - accuracy: 0.8504
Epoch 7/50 - loss: 0.4318 - accuracy: 0.8584
Epoch 8/50 - loss: 0.3985 - accuracy: 0.8693
Epoch 9/50 - loss: 0.3802 - accuracy: 0.8749
Epoch 10/50 - loss: 0.3621 - accuracy: 0.8827
Epoch 11/50 - loss: 0.3341 - accuracy: 0.8900
Epoch 12/50 - loss: 0.3168 - accuracy: 0.8956
Epoch 13/50 - loss: 0.2986 - accuracy: 0.9001
Epoch 14/50 - loss: 0.2896 - accuracy: 0.9031
Epoch 15/50 - loss: 0.2748 - accuracy: 0.9089
Epoch 16/50 - loss: 0.2638 - accuracy: 0.9109
Epoch 17/50 - loss: 0.2489 - accuracy: 0.9150
Epoch 18/50 - loss: 0.2393 - accuracy: 0.9173
Epoch 19/50 - loss: 0.2286 - accuracy: 0.9197
Epoch 20/50 - loss: 0.2256 - accuracy: 0.9228
Epoch 21/50 - loss: 0.